# UNIDAD IV — Manejo de archivos, persistencia y validación

## CLASE 3: Validaciones, errores y control de excepciones

---

# PARTE I — FUNDAMENTOS TEÓRICOS (Comprensión profunda)



---



# 1. ¿Qué es un error realmente?

Un error NO es solo que el programa falle.

Es una ruptura del contrato entre:

```
Lo que el programa espera
vs
Lo que realmente ocurre
```

---

## Tipos de errores en ingeniería de software

| Tipo                | Momento           | Ejemplo            |
| ------------------- | ----------------- | ------------------ |
| Sintáctico          | Antes de ejecutar | falta de :         |
| Lógico              | Funciona mal      | cálculo incorrecto |
| Runtime (excepción) | Durante ejecución | archivo no existe  |

Esta clase trata el tercero.

---

# 2. ¿Qué es una excepción?

Es un mecanismo controlado de fallo.

Python detiene la ejecución normal y busca un manejador.

Ejemplo:

```python
int("hola")
```

Produce:

```
ValueError
```

El programa no sabe continuar.

---

# 3. Filosofía profesional

Programación principiante:

> evitar errores

Programación profesional:

> asumir que los errores SIEMPRE ocurrirán

Por eso existen las excepciones.

---

# 4. try / except

Estructura básica:


In [ ]:
try:
    codigo_peligroso
except TipoError:
    recuperacion


---

## Ejemplo pedagógico


In [ ]:
try:
    edad = int(input("Edad: "))
except ValueError:
    print("Debe ingresar un número")



No evita el error — lo controla.

---


# 5. Múltiples excepciones



In [ ]:
try:
    archivo = open("datos.txt")
    numero = int(archivo.readline())
except FileNotFoundError:
    print("No existe el archivo")
except ValueError:
    print("Dato inválido")


---

# 6. Capturar sin saber el tipo (mala práctica)

```python
except:
```

Problema:

Oculta bugs graves.

Regla profesional:

> Nunca capturar excepciones genéricas sin razón

Correcto:

```python
except Exception as e:
    print(e)
```

---


# 7. else y finally

```python
try:
    lectura
except Error:
    manejar
else:
    si_todo_salio_bien
finally:
    siempre_se_ejecuta
```

---

## Uso real

| Bloque  | Propósito               |
| ------- | ----------------------- |
| try     | operación peligrosa     |
| except  | recuperación            |
| else    | lógica posterior segura |
| finally | liberar recursos        |

---



# 8. Errores comunes en archivos

---

## FileNotFoundError

Archivo inexistente

## PermissionError

Archivo bloqueado por otro programa

## UnicodeDecodeError

Codificación incorrecta

## ValueError

Formato inválido

## KeyError

Columna inexistente en CSV/dict

---


# 9. Validar vs manejar excepción

MUY IMPORTANTE:

No todo error debe atraparse.

Comparación:

| Estrategia    | Cuándo          |
| ------------- | --------------- |
| Validar antes | datos usuario   |
| Try/Except    | sistema externo |

---

Ejemplo:

MAL:

```python
try:
    edad = int(input())
except:
    pass
```

BIEN:

```python
dato = input()
if not dato.isdigit():
    print("Edad inválida")
```

---



# 10. Diseño de mensajes de error

El usuario no debe ver:

```
ValueError: invalid literal for int()
```

Debe ver:

```
Edad inválida. Ingrese un número entero.
```

Separar:

> Error técnico vs mensaje de usuario

---





---

# PARTE II — DISEÑO DE SOFTWARE ROBUSTO


---

## Capas de manejo de errores

| Capa            | Responsabilidad    |
| --------------- | ------------------ |
| Infraestructura | detecta error real |
| Repositorio     | traduce error      |
| Servicio        | decide acción      |
| Presentación    | mensaje limpio     |

---

Ejemplo flujo:

```
Archivo corrupto → ValueError
Repositorio → DataFormatError
Servicio → decide abortar carga
UI → "El archivo contiene datos inválidos"
```

---

## Principio fundamental

> Solo la capa superior habla con el usuario

---


---

# PARTE III — IMPLEMENTACIÓN GUIADA


Construiremos una aplicación segura de carga de datos.

---

# 1. Excepciones personalizadas

## core/errors.py

```python
class DataError(Exception):
    """Error general de datos"""

class DataFormatError(DataError):
    """Datos corruptos"""

class DataValidationError(DataError):
    """Datos inválidos"""

class StorageError(Exception):
    """Errores de persistencia"""
```

---




# 2. Validador profundo

## utils/validator.py


In [ ]:
from core.errors import DataValidationError

def validate_age(value: str) -> int:

    if not value.strip():
        raise DataValidationError("Campo vacío")

    if not value.isdigit():
        raise DataValidationError("Debe ser número entero")

    age = int(value)

    if age < 0 or age > 120:
        raise DataValidationError("Edad fuera de rango")

    return age


---

# 3. Infraestructura segura de archivos

## utils/safe_file.py



In [ ]:
from core.errors import StorageError

def safe_read(path: str) -> list[str]:

    try:
        with open(path, encoding="utf-8") as f:
            return f.readlines()

    except FileNotFoundError as e:
        raise StorageError("Archivo no encontrado") from e

    except PermissionError as e:
        raise StorageError("Sin permisos de lectura") from e

    except UnicodeDecodeError as e:
        raise StorageError("Error de codificación") from e


---

# 4. Repositorio

## repository/person_repository.py



In [ ]:
from utils.safe_file import safe_read
from core.errors import DataFormatError

class PersonRepository:

    def __init__(self, path: str):
        self.path = path

    def load(self):

        lines = safe_read(self.path)
        people = []

        for line in lines:
            parts = line.strip().split(",")

            if len(parts) != 2:
                raise DataFormatError("Línea corrupta")

            people.append(parts)

        return people


---

# 5. Servicio

## services/load_service.py



In [ ]:
from repository.person_repository import PersonRepository
from utils.validator import validate_age

class LoadService:

    def __init__(self, repo: PersonRepository):
        self.repo = repo

    def load_people(self):

        people = self.repo.load()
        clean = []

        for name, age in people:
            clean.append((name, validate_age(age)))

        return clean


---

# 6. Interfaz consola limpia

## main.py



In [ ]:
from services.load_service import LoadService
from repository.person_repository import PersonRepository
from core.errors import DataError, StorageError

def main():

    service = LoadService(PersonRepository("data/people.txt"))

    try:
        people = service.load_people()
        print("Datos cargados correctamente:")
        print(people)

    except StorageError as e:
        print("Error de acceso al archivo:", e)

    except DataError as e:
        print("Error en datos:", e)

    except Exception:
        print("Error inesperado. Contacte soporte.")


if __name__ == "__main__":
    main()


---

# TALLER PRÁCTICO (30 min)

Simular casos reales:

1. Archivo inexistente
2. Archivo con permisos denegados
3. Línea corrupta
4. Edad inválida
5. Archivo vacío
6. Encoding incorrecto

El programa NO debe colapsar en ninguno.

---
